In [ ]:
# Install hypertools (dev-1.0 preview) -- run this first on Colab.
# On release this becomes: %pip install hypertools
%pip install -q "hypertools[interactive] @ git+https://github.com/ContextLab/hypertools.git@dev-1.0"

%matplotlib inline

# Decades of weather: bold means, faint cities, temperature colormaps

This tutorial shows hierarchical data in one `hyp.plot` call. Monthly weather for several cities is reduced to 3-D; each city's decades trace a seasonal loop that slowly drifts. We group the cities by **hemisphere** and draw a bold hemisphere-*mean* loop per hemisphere plus the individual cities as faint context -- the classic bold-means / faint-leaves hierarchy -- smoothed and animated with **chemtrails**.

Every loop is colored by **temperature**, with a *separate* hot-cold colormap per hemisphere. We build the hierarchy as an explicit **list** of loops (city loops + hemisphere-mean loops) rather than a row `MultiIndex` DataFrame: hyp's MultiIndex expansion draws the same bold-means/faint-leaves hierarchy automatically, but it colors by *group* and ignores a continuous `hue=` (GH #95), so the temperature coloring would be lost. With a list, the per-point temperature `hue` applies. Emphasis (bold means / faint cities) is then set directly on the multicolor line collections by index.

Alongside the 3-D view we add a **second panel**: raw *daily* temperature for every city, one thin, translucent line per city under two bold hemisphere-mean lines, drawn with the *same* per-hemisphere colorscales and revealed in lockstep with the animation, with a vertical "now" cursor. The daily series are read back out of the same cached archive responses the monthly matrices were built from, so the panel costs no extra request. It is the same data seen the ordinary way, so the 3-D shape can be read against a familiar time series.

Data comes from the [open-meteo](https://open-meteo.com) archive (cached); if unavailable we synthesize seasonal loops and matching daily temperatures (hemispheres in opposite phase, with a warming drift).

## 1. Imports, cache, and cities

In [ ]:
import json, os, tempfile, urllib.request
import numpy as np
import pandas as pd
from matplotlib.cm import ScalarMappable
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, Normalize
from mpl_toolkits.mplot3d.art3d import Line3DCollection
import hypertools as hyp

CACHE = os.path.join(tempfile.gettempdir(), 'hypertools_tutorial')
os.makedirs(CACHE, exist_ok=True)
START, END = '1990-01-01', '2024-12-31'
CITIES = {
    'New York': (40.71, -74.01, 'Northern'),
    'London': (51.51, -0.13, 'Northern'),
    'Tokyo': (35.68, 139.69, 'Northern'),
    'Sydney': (-33.87, 151.21, 'Southern'),
    'Cape Town': (-33.92, 18.42, 'Southern'),
    'Santiago': (-33.45, -70.66, 'Southern'),
}
FEATS = ['temperature_2m_mean', 'precipitation_sum',
         'relative_humidity_2m_mean', 'windspeed_10m_max']

## 2. Fetch monthly means (with a synthetic fallback)

In [ ]:
def fetch_city_months(name, lat, lon):
    try:
        url = ('https://archive-api.open-meteo.com/v1/archive'
               f'?latitude={lat}&longitude={lon}'
               f'&start_date={START}&end_date={END}'
               f'&daily={",".join(FEATS)}&timezone=auto')
        dest = os.path.join(CACHE, f'wx_{name.replace(chr(32),chr(95))}.json')
        if not (os.path.exists(dest) and os.path.getsize(dest)):
            req = urllib.request.Request(
                url, headers={'User-Agent': 'ht-tutorial/1.0'})
            with urllib.request.urlopen(req, timeout=60) as r:
                open(dest, 'wb').write(r.read())
        d = json.load(open(dest))['daily']
        df = pd.DataFrame({f: pd.to_numeric(pd.Series(d[f]),
                           errors='coerce') for f in FEATS})
        df = df.interpolate().ffill().bfill()
        dt = pd.to_datetime(d['time'])
        df['ym'] = dt.year * 12 + dt.month
        return df.groupby('ym')[FEATS].mean().to_numpy()
    except Exception:
        return None


def synthetic_city_months(hemi, n_months=420, seed=0):
    rng = np.random.default_rng(seed)
    t = np.arange(n_months)
    phase = 0.0 if hemi == 'Northern' else np.pi
    season = np.sin(2 * np.pi * t / 12 + phase)
    warming = t / n_months
    temp = 14 + 11 * season + 3 * warming + rng.standard_normal(n_months) * 0.6
    precip = 60 + 25 * np.cos(2 * np.pi * t / 12 + phase) + rng.standard_normal(n_months) * 5
    humid = 70 + 10 * season + rng.standard_normal(n_months) * 2
    wind = 20 + 5 * np.sin(2 * np.pi * t / 12 + phase + 1.0) + rng.standard_normal(n_months) * 1.5
    return np.column_stack([temp, precip, humid, wind])


mats, hemis, offline = [], [], False
for seed, (name, (lat, lon, hemi)) in enumerate(CITIES.items()):
    m = fetch_city_months(name, lat, lon)
    if m is None:
        offline = True
        m = synthetic_city_months(hemi, seed=seed)
    mats.append(m)
    hemis.append(hemi)
print('offline fallback' if offline else 'open-meteo archive')

## 3. Reduce to 3-D and build a list of loops

We align run lengths and hand the whole list of cities to one `hyp.reduce(mats, reduce='IncrementalPCA', ndims=3, normalize='across')` call. `normalize='across'` z-scores the four weather features across the stacked rows of every city, one shared `IncrementalPCA` is fit on that stack, and the rows come back split into one array per city. With `ndims=3`, each month's weather features become a single 3-D point on the top 3 principal components; because the model is fit jointly across all cities, the resulting loops live in the same space and are directly comparable. From the per-city loops we build the two bold hemisphere-**mean** loops (and their mean temperatures). Everything stays a plain **list** of arrays -- no MultiIndex -- so the per-point temperature `hue` survives.

In [ ]:
min_len = min(len(m) for m in mats)
mats = [m[:min_len] for m in mats]
# ONE call on the LIST of cities: normalize='across' z-scores the stacked
# rows, a single IncrementalPCA is fit on that stack, and the rows are split
# back into one array per city -- no manual vstack, z-score, or slicing.
city_loops = [np.asarray(loop) for loop in
              hyp.reduce(mats, reduce='IncrementalPCA', ndims=3,
                         normalize='across')]
city_temp = [mats[i][:, 0] for i in range(len(mats))]

# hemisphere-mean loops (bold) from the per-city loops
N_idx = [i for i in range(len(mats)) if hemis[i] == 'Northern']
S_idx = [i for i in range(len(mats)) if hemis[i] == 'Southern']
Nmean_loop = np.mean([city_loops[i] for i in N_idx], axis=0)
Smean_loop = np.mean([city_loops[i] for i in S_idx], axis=0)
Nmean_temp = np.mean([city_temp[i] for i in N_idx], axis=0)
Smean_temp = np.mean([city_temp[i] for i in S_idx], axis=0)

## 4. A separate hot/cold colormap per hemisphere

We splice a northern (blue -> red) and a southern (teal -> amber) colormap into one `combined` palette, then map each hemisphere's temperatures onto its own half with `enc`. Because we pass a **list** of loops (not a MultiIndex), the per-point `hue` we build here is honored, so every loop is colored by temperature. The color axis is kept tight to the prominent hemisphere-**mean** range so the bold loops sweep the full colormap over the seasons; the faint cities that run hotter/colder saturate at the ends. The two per-hemisphere ranges also label the annotating colorbars.

In [ ]:
Ncm = LinearSegmentedColormap.from_list('N', ['#08306b', '#e31a1c'])  # blue->red
Scm = LinearSegmentedColormap.from_list('S', ['#0c7c8c', '#f4a300'])  # teal->amber
combined = LinearSegmentedColormap.from_list('combo',
    [Ncm(x) for x in np.linspace(0, 1, 128)]
    + [Scm(x) for x in np.linspace(0, 1, 128)])
Nlo, Nhi = float(Nmean_temp.min()), float(Nmean_temp.max())
Slo, Shi = float(Smean_temp.min()), float(Smean_temp.max())


def enc(temps, hemi):
    t = np.asarray(temps, float)
    if hemi == 'Northern':
        return np.clip(0.49 * (t - Nlo) / (Nhi - Nlo), 0.0, 0.49)     # [0, 0.49]
    return np.clip(0.51 + 0.49 * (t - Slo) / (Shi - Slo), 0.51, 1.0)  # [0.51, 1.0]


# datasets: faint city loops, then the two bold hemisphere-mean loops
datasets = list(city_loops) + [Nmean_loop, Smean_loop]
hue = np.concatenate(
    [enc(city_temp[i], hemis[i]) for i in range(len(mats))]
    + [enc(Nmean_temp, 'Northern'), enc(Smean_temp, 'Southern')])
CITY_LW, MEAN_LW = 1.0, 2.2      # means bold, but not heavy-handed
lws = [CITY_LW] * len(mats) + [MEAN_LW, MEAN_LW]

## 5. Plot: a list of loops + chemtrails + a temperature panel

We hand `hyp.plot` the list of loops with a per-point temperature `hue`, smoothed and animated with chemtrails. `manip='Smooth'` runs a Savitzky-Golay pass per dataset before anything is drawn, so each loop reads as a clean seasonal cycle instead of a noisy scribble; `chemtrails=True` leaves the loop already traversed glowing faintly behind each moving head, so the accumulated decades stay visible; and `legend=False, colorbar=False` suppress the library's own annotations because this figure draws its own two per-hemisphere colorbars (a single colorbar cannot describe two different colormaps). hyp's multicolor line collections don't inherit the per-dataset `linewidth`, and they reset per-segment alpha every frame, so we grab the `Line3DCollection`s and set the emphasis ourselves: the two hemisphere-mean loops are bold and opaque, the faint city loops recede. The collections are created head-first in dataset order, so the two means are the last two heads.

The figure is widened to hold a **second panel** on the right: every city's raw daily temperature, each drawn as a per-point-colored `LineCollection` using that city's *hemisphere* colormap and norm -- the same colorscales as the 3-D loops and the two colorbars. Two bold hemisphere-mean lines sit over the thin, translucent city lines, mirroring the bold-means / faint-leaves hierarchy of the 3-D view. The daily samples are laid out on the same x axis as the animation's month clock, so the panel is revealed in lockstep with the animation, with a vertical "now" cursor, and the familiar time series and the 3-D shape always show the same moment.

In [ ]:
# hyp.plot antialiases every drawn line by default (antialias=True), so these
# long seasonal loops render smooth at any frame rate
duration, fps = 8, 20
date_labels = pd.date_range(START, periods=min_len, freq='MS')

# THE hypertools call: a list of loops colored by a temperature hue, smoothed,
# chemtrails; the two bold means are emphasized below
fig, ani = hyp.plot(datasets, fmt='-', hue=hue, palette=combined,
                    colorbar=False, linewidth=lws, animate=True,
                    chemtrails=True, manip='Smooth', duration=duration,
                    frame_rate=fps, legend=False, elev=20, azim=-70,
                    size=(13, 6), show=False)
ax = [a for a in fig.axes if hasattr(a, 'zaxis')][0]
ax.set_position([-0.01, 0.03, 0.52, 0.90])       # 3-D view: left half

# hyp's multicolor collections don't inherit the per-dataset linewidth, so set
# the emphasis ourselves. Collections are created head-first in dataset order,
# so the first `NDS` are the per-dataset HEADS (the two means are the LAST two
# of those) and the rest are the faint trails.
_colls = [c for c in ax.collections if isinstance(c, Line3DCollection)]
NDS = len(datasets)
heads, trails = _colls[:NDS], _colls[NDS:]
MEAN_IDX = {NDS - 2, NDS - 1}
for k in MEAN_IDX:
    heads[k].set_linewidth(MEAN_LW)

# --- 2nd panel: every city's DAILY temperature ------------------------------
# The same hierarchy as the 3-D view, in ordinary axes: thin, translucent
# per-city lines (the "leaves") under two bold hemisphere-mean lines, every
# line colored point-by-point by that day's temperature through its OWN
# hemisphere colormap and norm -- the same two the colorbars label. It reveals
# in lockstep with the animation.


def fetch_city_daily_temp(name):
    """Daily mean temperature for one city, or None.

    Read back out of the SAME cached archive response fetch_city_months built
    its monthly matrix from, so this panel costs no extra request.
    """
    try:
        dest = os.path.join(CACHE, f'wx_{name.replace(chr(32),chr(95))}.json')
        d = json.load(open(dest))['daily']
        v = pd.to_numeric(pd.Series(d['temperature_2m_mean']), errors='coerce')
        return v.interpolate().ffill().bfill().to_numpy(dtype=float)
    except Exception:
        return None


def synthetic_city_daily(hemi, n_days, seed=0):
    """Fallback: the daily-resolution twin of synthetic_city_months."""
    rng = np.random.default_rng(seed)
    t = np.arange(n_days)
    phase = 0.0 if hemi == 'Northern' else np.pi          # opposite seasons
    season = np.sin(2 * np.pi * t / 365.25 + phase)
    return 14 + 11 * season + 3 * (t / n_days) + rng.standard_normal(n_days) * 1.5


daily = []
for seed, (name, (_, _, hemi)) in enumerate(CITIES.items()):
    v = fetch_city_daily_temp(name)
    if v is None:
        v = synthetic_city_daily(hemi, int(round(min_len * 365.25 / 12)),
                                 seed=seed)
    daily.append(v)
ND = min(len(v) for v in daily)
daily = [v[:ND] for v in daily]
# daily samples laid out on the SAME x axis as the animation's month clock, so
# the reveal cursor and the month index line up exactly
day_x = np.linspace(0, min_len - 1, ND)
Nmean_daily = np.mean([daily[i] for i in N_idx], axis=0)
Smean_daily = np.mean([daily[i] for i in S_idx], axis=0)

ax_t = fig.add_axes([0.575, 0.145, 0.30, 0.70])
nrmN, nrmS = Normalize(Nlo, Nhi), Normalize(Slo, Shi)
temp_colls = []


def temp_line(y, hemi, lw, alpha, z):
    """One per-day-colored temperature line, added hidden (revealed below).

    The weights are deliberately far apart. Decades of seasonal cycles land
    only a few pixels apart across this panel, so every line here is a dense
    picket fence rather than a followable curve; the means separate from the
    city haze only if the cities are drawn very faint and very thin. (A white
    halo under the means was tried and removed: at this cycle density the halo
    fills the band with white instead of outlining a curve.)
    """
    pts = np.column_stack([day_x, y]).reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc = LineCollection(segs, cmap=Ncm if hemi == 'Northern' else Scm,
                        norm=nrmN if hemi == 'Northern' else nrmS,
                        linewidth=lw, alpha=alpha, zorder=z)
    lc.set_segments([])                        # revealed progressively below
    ax_t.add_collection(lc)
    temp_colls.append((lc, segs, y[:-1]))      # y[:-1] = per-segment temp


for i in range(len(mats)):                     # faint city "leaves"
    temp_line(daily[i], hemis[i], 0.3, 0.12, 2)
temp_line(Nmean_daily, 'Northern', 1.8, 1.0, 4)    # bold hemisphere means,
temp_line(Smean_daily, 'Southern', 1.8, 1.0, 4)    # opaque, as in the 3-D view
_allt = np.concatenate(daily)
ax_t.set_xlim(0, min_len - 1)
ax_t.set_ylim(_allt.min() - 1.5, _allt.max() + 1.5)
_yr_ticks = list(range(0, min_len, 60))                    # every 5 years
ax_t.set_xticks(_yr_ticks)
ax_t.set_xticklabels([date_labels[i].strftime('%Y') for i in _yr_ticks],
                     fontsize=9)
ax_t.tick_params(axis='y', labelsize=9)
ax_t.set_ylabel('daily mean temperature (°C)', fontsize=10)
ax_t.set_title('every city, every day', fontsize=11, color='#333', pad=6)
for _s in ('top', 'right'):
    ax_t.spines[_s].set_visible(False)
ax_t.grid(alpha=0.18, linewidth=0.6)
now_line = ax_t.axvline(0, color='#444', lw=1.2, alpha=0.8)   # "now" cursor

caxN = fig.add_axes([0.925, 0.55, 0.016, 0.33])
fig.colorbar(ScalarMappable(Normalize(Nlo, Nhi), Ncm),
             cax=caxN).set_label('Northern temp (°C)', fontsize=9)
caxS = fig.add_axes([0.925, 0.145, 0.016, 0.33])
fig.colorbar(ScalarMappable(Normalize(Slo, Shi), Scm),
             cax=caxS).set_label('Southern temp (°C)', fontsize=9)

# the title spans the WHOLE figure (3-D box + temperature panel), not the box
title = fig.text(0.47, 0.965, '', ha='center', va='top',
                 fontsize=13.5, fontweight='bold', color='#1a1a1a')
total = int(round(fps * duration))
_orig = ani._func


def _wrapped(frame, *args):
    result = _orig(frame, *args)
    # bold means opaque, faint cities receded -- re-applied each frame since the
    # multicolor updater re-sets per-segment colors (which resets alpha)
    for k, c in enumerate(heads):
        c.set_alpha(1.0 if k in MEAN_IDX else 0.16)
    for c in trails:
        c.set_alpha(0.10)
    idx = min(min_len - 1, int(frame / max(1, total - 1) * min_len))
    # 2nd panel reveals in lockstep with the 3-D animation (month -> day index)
    kd = int(np.clip(round(idx / max(1, min_len - 1) * (ND - 1)), 0, ND - 1))
    for lc, segs, vals in temp_colls:
        lc.set_segments(segs[:kd])
        lc.set_array(vals[:kd])            # keep colors aligned to segments
    now_line.set_xdata([idx, idx])
    title.set_text('decades of weather, 6 cities  '
                   + date_labels[idx].strftime('%b %Y'))
    return result


ani._func = _wrapped

## 6. Display the animation

In [ ]:
from IPython.display import HTML
HTML(ani.to_jshtml())